In [ ]:
# imports
import pandas as pd
from pathlib import Path
import requests
import zipfile
import os
import r5py
import geopandas as gpd
from datetime import datetime, timedelta
from shapely.geometry import Point

In [3]:
# Configuration of paths
URL = "https://data.opentransportdata.swiss/fr/dataset/timetable-2026-gtfs2020/permalink"
GTFS_DIR = Path("data/gtfs")
ZIP_FILE = GTFS_DIR / "gtfs_fp2026.zip"

# Create the directory if it doesn't exist
GTFS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# Download the ZIP file if it's not already present
if not ZIP_FILE.exists():
    print(f"Downloading GTFS data to {ZIP_FILE}...")
    # Stream the download to handle large files efficiently
    r = requests.get(URL, stream=True)
    r.raise_for_status() 
    with open(ZIP_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [5]:
# Extracting all files directly into data/gtfs/
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(GTFS_DIR)
    print(f"Files extracted to {GTFS_DIR}")

Files extracted to data/gtfs


In [11]:
# --- Configuration for OSM data ---
OSM_URL = "https://download.geofabrik.de/europe/switzerland-latest.osm.pbf"
OSM_FILE = Path("data/switzerland-latest.osm.pbf")

# Download the PBF file if it's not already present
if not OSM_FILE.exists():
    print(f"Downloading OSM data to {OSM_FILE} (this may take a while)...")
    # Stream the download
    r = requests.get(OSM_URL, stream=True)
    r.raise_for_status()
    
    # On ajoute une petite barre de progression textuelle simple
    total_size = int(r.headers.get('content-length', 0))
    downloaded = 0
    
    with open(OSM_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024): # 1MB chunks
            f.write(chunk)
            downloaded += len(chunk)
            print(f"\rProgress: {downloaded/(1024*1024):.1f}/{total_size/(1024*1024):.1f} MB", end="")
    print(f"\n✅ OSM data downloaded to {OSM_FILE}")
else:
    print(f"✅ OSM file already exists at {OSM_FILE}")

Progress: 502.0/502.0 MB
✅ OSM data downloaded to data/switzerland-latest.osm.pbf


In [6]:
# Loading the DataFrames
stops = pd.read_csv(GTFS_DIR / "stops.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
print("DataFrames loaded successfully.")

/var/folders/dg/jjw_9gyd1p38f4_jp_6s89r00000gn/T/ipykernel_54453/470310557.py:2: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  stops = pd.read_csv(GTFS_DIR / "stops.txt")


DataFrames loaded successfully.


In [7]:
# We look for routes that are InterCity (IC) trains based on their route_desc 'IC'
ic_routes = routes[routes['route_desc'] == 'IC']
ic_trips = trips[trips['route_id'].isin(ic_routes['route_id'])]

# Get the stop_id where IC trains actually stop
ic_stop_ids = stop_times[stop_times['trip_id'].isin(ic_trips['trip_id'])]['stop_id'].unique()
ic_stops = stops[stops['stop_id'].isin(ic_stop_ids)]

print(f"Number of IC stops identified: {len(ic_stops)}")

Number of IC stops identified: 430


In [8]:
# Test de la présence de Java (indispensable pour r5py)
java_check = os.popen('java -version 2>&1').read()

if "version" in java_check.lower():
    print(f"✅ Succès : Java est bien lié à l'env 'dataviz'.\nDétails : {java_check.splitlines()[0]}")
    print(f"✅ r5py est prêt à l'emploi.")
else:
    print("❌ Erreur : Java n'est pas détecté dans cet environnement.")
    print("Conseil : Tape 'conda install -c conda-forge openjdk' dans ton terminal.")

✅ Succès : Java est bien lié à l'env 'dataviz'.
Détails : openjdk version "25.0.1" 2025-10-21 LTS
✅ r5py est prêt à l'emploi.


In [12]:
# Initialiser le réseau (Longue étape, car il compile le graphe)
network = r5py.TransportNetwork(
    str(OSM_FILE),
    [str(ZIP_FILE)]
)

: 

In [ ]:
# Configurer le calcul
computer = r5py.TravelTimeMatrixComputer(
    network,
    origins=ma_grille_de_points,      # GeoDataFrame de points (EPSG:4326)
    destinations=mes_gares_ic,        # GeoDataFrame des hubs IC
    departure=datetime(2026, 3, 20, 8, 0), 
    departure_time_window=timedelta(minutes=60), # Fenêtre d'une heure
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    percentiles=[50] # On prend la médiane pour être robuste
)

In [ ]:
# Exécuter
travel_times = computer.compute_travel_times()

In [ ]:
travel_times.shape()